# Committor Model: Non-Parametric Approach (toytis long MD)

Trains a non-parametric committor on long MD trajectories from **toytis**: either `md_run.py` output
(one `md_path.txt` per seed, columns `step x1 .. v1 ..`) or an older toytis run saved as `md_path.npy`.
The model predicts the probability of reaching state B (`lambda >= LAMBDA_B`) before state A
(`lambda <= LAMBDA_A`), where `lambda` is the order parameter (the first position for toytis runs).

**Method:** NPq (`optimalrcs.Committor`, equilibrium data) by default; NPNEq (`optimalrcs.CommittorNE`,
with the history features of the original notebook) as an option. Uses the CuPy implementation.

**Data:** set `MD_FILES` and the related options in section 2. Several files (seeds) are separate
trajectories; nothing is fitted across the jump from one file to the next.

## 1. Imports

In [2]:
import numpy as np
import os
import glob
import math
import importlib
import importlib.util
import sys
import pandas as pd
import cupy as cp
import matplotlib.pyplot as plt

# Plot styling
plt.rc('font', size=10)
plt.rc('axes', titlesize=14)
plt.rc('axes', labelsize=14)
plt.rc('xtick', labelsize=12)
plt.rc('ytick', labelsize=12)
plt.rc('legend', fontsize=12)

# Custom optimal-RC / committor package (CuPy version in ../optimalrcs-cupy).
# Its optimalrcs.py shares its name with the TensorFlow package of this repository, which may also be
# importable, so it is loaded from its file under a name of its own; the other modules are flat imports.
CUPY_DIR = os.path.abspath(os.path.join(os.getcwd(), os.pardir, 'optimalrcs-cupy'))
sys.path.insert(0, CUPY_DIR)
_spec = importlib.util.spec_from_file_location('optimalrcs_cupy', os.path.join(CUPY_DIR, 'optimalrcs.py'))
optimalrcs = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(optimalrcs)
import metrics, plots

## 2. Data Loading

Reads one or more toytis long MD files in chunks. Per file, the first `BURN_IN` saved rows are dropped
(relaxation from the start point) and every `EVERY`-th row is kept.

With `EXCURSIONS_ONLY` only the rows outside A and B are kept, together with the A/B row right before
and after each excursion; every excursion becomes its own trajectory segment. For NPq this loses
nothing: a step between two rows inside the same state has a fixed committor (0 or 1) and contributes
nothing to the fit. It matters for systems that spend most of their time inside a state (e.g. the 2D Z
potential), where it keeps the data small enough for the GPU. A histogram of `lambda` over all rows is
kept for the equilibrium free energy profile (section 3).

In [3]:
# --- Which long MD(s) to read -------------------------------------------------------------
TOYTIS = '/mnt/0bf0c339-34bb-4500-a5fb-f3c2a863de29/DATA/PyRETIS3/toytis'
# md_run.py output, one file per seed, e.g.:
#   MD_FILES = sorted(glob.glob(f'{TOYTIS}/simulations/md_zbr_g10_s*/md_path.txt'))
# or an older toytis run saved as .npy (no header: give its columns in NPY_COLUMNS):
MD_FILES = [f'{TOYTIS}/simulations/1dwell_longmd_flatbr50M/md_path.npy']   # 1D cosine well, underdamped
NPY_COLUMNS = ['x1', 'v1']        # meaning of the .npy columns, e.g. ['x1'] for the 1D Brownian runs
ORDER_COL = 'x1'                  # column of the order parameter lambda
LAMBDA_A, LAMBDA_B = -0.35, 0.35  # A: lambda <= LAMBDA_A, B: lambda >= LAMBDA_B
BURN_IN = 50_000                  # saved rows dropped at the start of every file
EVERY = 1                         # keep every n-th saved row (on top of md_run's --stride)
EXCURSIONS_ONLY = True            # keep only rows outside A/B plus their A/B neighbours (see above)
USE_VELOCITIES = True             # velocities as CVs: only for underdamped runs that saved them
CHUNK = 5_000_000                 # rows per read chunk
HIST_EDGES = np.linspace(LAMBDA_A - 0.3 * (LAMBDA_B - LAMBDA_A), LAMBDA_B + 0.3 * (LAMBDA_B - LAMBDA_A), 201)


def iter_chunks(path):
    """Yield (row index, rows, column names) chunks of one toytis long MD file."""
    if path.endswith('.npy'):
        arr = np.load(path, mmap_mode='r')
        arr = arr.reshape(len(arr), -1)
        names = list(NPY_COLUMNS)
        if arr.shape[1] != len(names):
            raise ValueError(f'{path} has {arr.shape[1]} columns but NPY_COLUMNS lists {len(names)}')
        for s in range(0, len(arr), CHUNK):
            rows = np.asarray(arr[s:s + CHUNK], dtype=np.float64)
            yield np.arange(s, s + len(rows)), rows, names
    else:
        with open(path) as f:
            names = f.readline().lstrip('#').split()     # md_run.py header: step x1 .. v1 ..
        start = 0
        for df in pd.read_csv(path, sep=r'\s+', comment='#', header=None, names=names,
                              chunksize=CHUNK, dtype=np.float64):
            rows = df.to_numpy()
            yield np.arange(start, start + len(rows)), rows, names
            start += len(rows)


def load_md(path):
    """Kept rows of one file, their row indices, the column names and a lambda histogram of all rows."""
    kept_rows, kept_idx, hist, names = [], [], np.zeros(len(HIST_EDGES) - 1), None
    pending = None    # last row of the previous chunk, whose right neighbour is not read yet
    for idx, rows, names in iter_chunks(path):
        sel = (idx >= BURN_IN) & ((idx - BURN_IN) % EVERY == 0)
        idx, rows = idx[sel], rows[sel]
        if len(idx) == 0:
            continue
        lam = rows[:, names.index(ORDER_COL)]
        hist += np.histogram(lam, HIST_EDGES)[0]
        if not EXCURSIONS_ONLY:
            kept_rows.append(rows); kept_idx.append(idx)
            continue
        outside = (lam > LAMBDA_A) & (lam < LAMBDA_B)
        left_of_first = False
        if pending is not None:
            rows, idx = np.vstack([pending[0], rows]), np.r_[pending[1], idx]
            left_of_first, outside = pending[3], np.r_[pending[2], outside]
        left = np.r_[left_of_first, outside[:-1]]
        right = np.r_[outside[1:], False]
        keep = outside | left | right
        kept_rows.append(rows[:-1][keep[:-1]]); kept_idx.append(idx[:-1][keep[:-1]])
        pending = (rows[-1:], idx[-1:], outside[-1:], bool(outside[-2]) if len(outside) > 1 else left_of_first)
    if pending is not None and (pending[2][0] or pending[3]):
        kept_rows.append(pending[0]); kept_idx.append(pending[1])
    return np.concatenate(kept_rows), np.concatenate(kept_idx), names, hist


data, row_idx, file_id, lam_hist, names = [], [], [], 0, None
for k, path in enumerate(MD_FILES):
    rows, idx, names, hist = load_md(path)
    data.append(rows); row_idx.append(idx); file_id.append(np.full(len(idx), k)); lam_hist = lam_hist + hist
    print(f'{path}: {int(hist.sum())} rows after burn-in, kept {len(idx)}')
data, row_idx, file_id = np.concatenate(data), np.concatenate(row_idx), np.concatenate(file_id)
lam = data[:, names.index(ORDER_COL)]
cv_names = [n for n in names if n.startswith('x') or (USE_VELOCITIES and n.startswith('v'))]
X_cv = data[:, [names.index(n) for n in cv_names]]
print(f'Kept {len(lam)} rows; CVs: {cv_names}')
if len(lam) > 8_000_000:
    print('Warning: more than ~8M rows may not fit on an 8 GB GPU; use EXCURSIONS_ONLY, EVERY or fewer files.')

/mnt/0bf0c339-34bb-4500-a5fb-f3c2a863de29/DATA/PyRETIS3/toytis/simulations/1dwell_longmd_flatbr50M/md_path.npy: 47644837 rows after burn-in, kept 35112940
Kept 35112940 rows; CVs: ['x1', 'v1']


## 3. Free Energy Profile (FEP)

Equilibrium FEP of the order parameter, from the histogram over all rows after burn-in (not only the
kept ones), to visualize the two states and the barrier between them.

In [ ]:
centers = (HIST_EDGES[1:] + HIST_EDGES[:-1]) / 2
ok = lam_hist > 0
fig, ax = plt.subplots(1, 1)
ax.plot(centers[ok], -np.log(lam_hist[ok] / lam_hist.sum()), 'r-')
for lam_b in (LAMBDA_A, LAMBDA_B):
    ax.axvline(lam_b, color='k', ls='--', lw=0.8)
ax.set(xlabel='$\\lambda$', ylabel='$F/kT$')
ax.grid()
plt.show()

## 4. Trajectory Indices

Defines trajectory indexing for the committor model. Every file is its own trajectory, and with
`EXCURSIONS_ONLY` every excursion out of A or B is its own segment: a new segment starts wherever two
kept rows were not consecutive in their file.

Because every excursion starts and ends in A or B, section 5 calls `set_boundary_traps()`: segment edges
in A/B are then not treated as cuts, and the Z_q profiles equal those of the continuous trajectory
(without it Z_q drops at large lags). For the exact committor Z_q(q, Δt) = N_AB, the number of A→B
transitions, at every q and Δt.

In [ ]:
new_segment = np.r_[True, (file_id[1:] != file_id[:-1]) | (np.diff(row_idx) != EVERY)]
i_traj = np.cumsum(new_segment)        # trajectory (segment) index
t_traj = row_idx.astype(np.float64)    # saved-row index within its file
print(f'Frames: {len(lam)}, segments: {i_traj[-1]}, files: {len(MD_FILES)}')
print(f'Frames in A: {(lam <= LAMBDA_A).sum()}, in B: {(lam >= LAMBDA_B).sum()}')

## 5. Train Committor (Non-Parametric)

Trains the committor over 10 random seeds. Each iteration uses a randomly chosen CV (a position, or a
velocity with `USE_VELOCITIES`) as the observed variable `y`.

`METHOD = 'NPq'` uses `optimalrcs.Committor`, the equilibrium estimator; it has no history features, so
`history`, `yr_type` and `shift_type` are not used. `METHOD = 'NPNEq'` uses `optimalrcs.CommittorNE` with
the history features below, as in the original notebook.

**Parameters:**
| Parameter | Value | Description |
|-----------|-------|-------------|
| `envelope_scale` | 0.001 | RC envelope width |
| `history` | 0..10 | History lag times (NPNEq only) |
| `gamma` | 0.2 | RC regularization |
| `max_iter` | 45000 | Maximum iterations |
| `history_type` | `y(t-d),r(t-d)` | Feature types (NPNEq only) |
| `history_shift_type` | `r(t)` | Shift reference (NPNEq only) |

**Seeds and warm-up** (`seed_runs.run_seeds`, shared with the other notebooks): the fit is repeated for every seed in `SEEDS` (the seed sets which CV and envelope centre each iteration draws). The first `WARMUP_ITER` iterations of every seed can use their own envelope (`WARMUP_ENVELOPE`), gamma (`WARMUP_GAMMA`) and CV (`WARMUP_CV`: `None` random, otherwise the index of the CV to use); then `ENVELOPE` and `GAMMA` take over (`WARMUP_ITER = 0`: no warm-up). After the warm-up and every `CHUNK` iterations the plots in `PLOTS` are shown here and saved to `<OUT>/seed<s>/it<N>_<plot>.png`, with a progress line. The objects of earlier seeds are freed; `q` is the last seed's fit and `results[seed]['r']` every seed's RC.

In [ ]:
import seed_runs

# Configuration
ENVELOPE = 0.001                   # envelope_scale
history = list(range(11))          # lag times 0..10
GAMMA = 0.2                        # regularization strength
MAX_ITER = 45000                   # iterations per seed (after the warm-up)
SEEDS = list(range(10))            # random seeds
CHUNK = 5000                       # iterations between shown and saved plots
WARMUP_ITER = 0                    # optional warm-up (0 = off)
WARMUP_ENVELOPE = ENVELOPE
WARMUP_GAMMA = GAMMA
WARMUP_CV = None                   # None: random CV; otherwise the index of the CV to use
PLOTS = ('metrics', 'feps', 'obs_pred', 'obs_pred_log')

# Feature specification (NPNEq only)
yr_type = ['y(t-d),r(t-d)']       # observed var + history features
shift_type = 'r(t)'                # shift reference

METHOD = 'NPq'                     # 'NPq' (optimalrcs.Committor) or 'NPNEq' (optimalrcs.CommittorNE)
OUT = f'figures/nb_md/{METHOD}'

# CVs on the GPU once; a random one is the observed variable in each iteration
Y = cp.asarray(X_cv)
def comp_y():
    return Y[:, np.random.randint(Y.shape[1])]

boundary0 = lam <= LAMBDA_A        # state A: q = 0
boundary1 = lam >= LAMBDA_B        # state B: q = 1

def make_q():
    cls = optimalrcs.Committor if METHOD == 'NPq' else optimalrcs.CommittorNE
    q = cls(boundary0=boundary0, boundary1=boundary1, i_traj=i_traj, t_traj=t_traj)
    if EXCURSIONS_ONLY:
        q.set_boundary_traps()   # segments start/end in A or B: Z_q as for the continuous run
    return q

fit_kw = {} if METHOD == 'NPq' else dict(history_delta_t=history, history_type=yr_type, history_shift_type=shift_type)
results, q = seed_runs.run_seeds(
    optimalrcs, make_q, comp_y, fit_kw,
    seeds=SEEDS, max_iter=MAX_ITER, chunk=CHUNK, out_dir=OUT, envelope=ENVELOPE, gamma=GAMMA,
    warmup_iter=WARMUP_ITER, warmup_envelope=WARMUP_ENVELOPE, warmup_gamma=WARMUP_GAMMA,
    warmup_comp_y=None if WARMUP_CV is None else (lambda: Y[:, WARMUP_CV]),
    plots=PLOTS,
)

### 5b. Committor on the coordinates

The fitted committor (last seed) against the order parameter (1D), on (x, v) when velocities are CVs,
or on the two positions (2D). Optionally with the contours of a toytis potential (`POTENTIAL`); for 2D
toytis potentials the positions are ordered (order parameter, transverse).

In [ ]:
POTENTIAL = None   # e.g. ('sjoelbak_angle:RectangularGridWithAngledBarrierPotential', dict(Lx=0.3, angle=45))
                   #      ('cos_well:CosWellWalls', {})
r = cp.asnumpy(q.r_traj)
pos = [n for n in names if n.startswith('x')]
show = np.random.default_rng(0).permutation(len(r))[:200_000]

def load_potential(spec, kwargs):
    sys.path.insert(0, TOYTIS)
    module, cls = spec.split(':')
    return getattr(importlib.import_module('potentials.' + module), cls)(**kwargs)

fig, ax = plt.subplots(figsize=(8, 4.5))
if len(pos) == 1 and not (USE_VELOCITIES and 'v1' in names):
    ax.scatter(lam[show], r[show], s=1, alpha=0.3, edgecolors='none')
    ax.set(xlabel='$\\lambda$', ylabel='committor')
    if POTENTIAL is not None:
        pot = load_potential(*POTENTIAL)
        grid = np.linspace(lam.min(), lam.max(), 400)
        ax2 = ax.twinx()
        ax2.plot(grid, [pot.potential_and_force((g, 0.0))[0] for g in grid], color='gray')
        ax2.set_ylabel('potential')
else:
    cx = names.index(pos[0])
    cy = names.index('v1') if len(pos) == 1 else names.index(pos[1])
    if POTENTIAL is not None and len(pos) == 2:
        pot = load_potential(*POTENTIAL)
        gx = np.linspace(data[:, cx].min(), data[:, cx].max(), 150)
        gy = np.linspace(data[:, cy].min(), data[:, cy].max(), 100)
        f = getattr(pot, 'effective_potential', None) or (lambda ph: pot.potential_and_force(ph)[0])
        V = np.array([[f((np.array([a, b]), np.zeros(2))) for a in gx] for b in gy])
        ax.contour(gx, gy, V, levels=np.linspace(*np.percentile(V, [1, 95]), 15), colors='0.5', linewidths=0.6)
    sc = ax.scatter(data[show, cx], data[show, cy], c=r[show], cmap='coolwarm', s=2, edgecolors='none',
                    vmin=0, vmax=1)
    fig.colorbar(sc, ax=ax, label='committor')
    ax.set(xlabel=names[cx], ylabel=names[cy])
plt.show()

## 6. Save Results

Uncomment to save the raw committor trajectory and its normalized version to disk.

In [ ]:
# Save raw committor values
# np.save('q_traj.npy', cp.asnumpy(q.r_traj))

# Save normalized committor
# qn_traj = plots.transform_q2qn(q.r_traj)
# np.save('qn_traj.npy', cp.asnumpy(qn_traj))

## 7. Recommendations for toytis long MDs

The training parameters above (`history` 0..10, `gamma` 0.2, `max_iter` 45000, 10 seeds) come from the
protein example. For the toytis systems the CVs below are the **full state**, so the dynamics in them is
Markovian and no history is needed; NPq is the natural method.

| System | Data settings | Fit | Notes |
|---|---|---|---|
| **1D Brownian** (flat, two bumps; `md_path.npy` with only `lambda`, gamma 5) | `NPY_COLUMNS=['x1']`, `USE_VELOCITIES=False`, `LAMBDA_A, LAMBDA_B = -0.1, 0.3` | NPq | Converged in ~1500 iterations (`\|dx\|` ~1e-5) on 5M-frame blocks; one seed is enough. A/B hold ~53% of 50M frames, so `EXCURSIONS_ONLY` still keeps ~23M: use `EVERY` or blocks to stay under ~8M rows |
| **1D Langevin** (cosine well; `md_path.npy` with x and v) | `NPY_COLUMNS=['x1','v1']`, `USE_VELOCITIES=True`, `LAMBDA_A, LAMBDA_B = -0.35, 0.35` | NPq on (x, v) | With the recorded velocity a standard lag-1 fit is accurate (on the iSTAR well data: RMS 0.019 vs a brute-force reference). Without velocities x alone is not Markovian: estimate v from positions and fit at lag ~4 (see `docs/npneq_tis_notes.md` section 10) |
| **2D Brownian** (Z potential; `md_run.py --high-friction`, 16 seeds) | `MD_FILES` = the 16 `md_path.txt`, `USE_VELOCITIES=False` (v columns are noise), `LAMBDA_A, LAMBDA_B = 0.05, 0.85`, `EXCURSIONS_ONLY=True` | NPq on the two positions | The system sits in A >99% of the time, so `EXCURSIONS_ONLY` is what makes it fit. Exact reference: `figures/z_brownian/committor_reference_h0.001.npz`. Plot with `POTENTIAL = ('sjoelbak_angle:RectangularGridWithAngledBarrierPotential', dict(Lx=0.3, angle=45))` |
| **2D Langevin** | positions and velocities, `USE_VELOCITIES=True` | NPq on 4 CVs | The committor depends jointly on positions and velocities: expect slower convergence; try `ny` 6-8 and more iterations |

General:
- **Iterations and seeds:** watch `|dx|`; 2000-5000 iterations were enough for 1D and 2D toy data. Training
  seeds only change the order in which CVs are drawn; independent MD seeds or blocks give the real error bars.
- **Saved-row spacing:** A/B are detected on saved rows only. With `md_run --stride s` (and `EVERY`), the
  effective boundaries move out by ~0.58 sigma sqrt(s * EVERY) (sigma = Brownian step); use the same shift when
  comparing with a reference.
- **GPU memory:** `Committor` builds the polynomial basis for all frames: about 28 rows x frames x 8 bytes,
  several times over; ~5-8M frames fit on 8 GB.
- **Burn-in:** the toytis MDs start from one point; `BURN_IN` = a few 10^4-10^5 saved rows removes the initial
  relaxation, which matters for NPq (equilibrium data).